# 03 Features: one row per monotown

This notebook builds the variables used for clustering in notebook 04.
For each of the 247 matched monotowns we calculate:

- population 2025 and population change 2017-2025 (%)
- net migration per 1,000 residents (average of 2021-2024)
- average monthly wage 2025
- jobs per 1,000 residents 2025 and job change 2017-2025 (%)
- industry dependence: share of jobs in mining and manufacturing
- English names for towns, regions and municipality types

Input: the cleaned files from `data/processed/` (notebook 01) and the matched list `monotowns_oktmo.csv` (notebook 02).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED = Path("../data/processed")
pd.set_option("display.max_columns", 50)

## Load the data and check the loaded files

OKTMO is the Russian code for municipalities (like a postcode for local governments).
We read it as text, not as a number, because some codes start with 0 (e.g. 04712000 for Yeniseysk),
and reading them as numbers would silently drop that zero and break the matching.

In [2]:
indicator_files = ["population", "migration", "wages", "employees", "industry", "profitable"]

data = {
    name: pd.read_csv(PROCESSED / f"{name}.csv", dtype={"oktmo": str, "oktmo_stable": str})
    for name in indicator_files
}

mono = pd.read_csv(PROCESSED / "monotowns_oktmo.csv", dtype={"oktmo": str})

In [3]:
for name, df in data.items():
    years = sorted(df["year"].unique())
    lengths = df["oktmo"].str.len().value_counts().to_dict()
    print(f"{name:11} rows={len(df):>7}  years={years}  oktmo lengths={lengths}")

print()
for name in ["population", "migration", "wages", "employees", "profitable"]:
    dups = data[name].duplicated(["oktmo", "year"]).sum()
    print(f"{name:11} duplicates (oktmo, year): {dups}")

population  rows=  61486  years=[np.int64(2017), np.int64(2022), np.int64(2023), np.int64(2025)]  oktmo lengths={8: 61486}
migration   rows=  58560  years=[np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]  oktmo lengths={8: 58560}
wages       rows=   2486  years=[np.int64(2025)]  oktmo lengths={8: 2486}
employees   rows=   4927  years=[np.int64(2017), np.int64(2025)]  oktmo lengths={8: 4927}
industry    rows=   3187  years=[np.int64(2017), np.int64(2025)]  oktmo lengths={8: 3187}
profitable  rows=  13476  years=[np.int64(2024)]  oktmo lengths={8: 13476}

population  duplicates (oktmo, year): 0
migration   duplicates (oktmo, year): 0
wages       duplicates (oktmo, year): 0
employees   duplicates (oktmo, year): 0
profitable  duplicates (oktmo, year): 0


In [4]:
print(mono["note"].value_counts(), "\n")

matched = mono[mono["oktmo"].notna()].copy()
print("matched towns:", len(matched))                          # expected 247
print("duplicated oktmo:", matched["oktmo"].duplicated().sum())  # expected 0
print("oktmo lengths:", matched["oktmo"].str.len().value_counts().to_dict())

note
automatic                                       213
manual correction                                15
excluded: region not in Rosstat data              9
matched via migration table                       9
excluded: closed city (ZATO), no public data      7
district proxy                                    6
city part of merged okrug                         4
excluded: district proxy not representative       1
Name: count, dtype: int64 

matched towns: 247
duplicated oktmo: 0
oktmo lengths: {8: 247}


## Population and population change

Population is pivoted from one row per year to one row per town (`pop_2017`, `pop_2022`, `pop_2023`, `pop_2025`) and joined
to the 247 monotowns by OKTMO. Change is in percent: -15 means the town lost 15% of its residents.

Rosstat's yearly population estimates start from the last census and add registered births, deaths and moves. The 2021 census reset this count: estimates up to 1 January 2022 are still based on the 2010 census, and from 1 January 2023 on the 2021 census. Comparing years across this break mixes real change with the census correction. So three variables are computed:
- `pop_change_pct`: 2017 to 2025, a longer period but it mixes the two baselines.
- `pop_change_23_25`: 2023 to 2025, only two years but both estimates are based on the 2021 census.
- `census_jump_pct`: 2022 to 2023, the size of the census correction (for example, about 2,500 extra people in Slantsy without anyone moving in).

Notebook 04 compares the two population changes and chooses one for clustering.

Some towns are expected to lack values: those matched via the migration table (Crimea, Buryatia
and Dagestan are missing from the 2025 file) and municipalities whose OKTMO code changed between
the years (e.g. a district turned into an okrug). The dataset's `oktmo_stable` column links the
same territory across code changes, so it is used to fill these gaps; settlements still missing get their district's change as a proxy.

In [5]:
pop = data["population"]

pop_wide = (
    pop.pivot(index="oktmo", columns="year", values="indicator_value")
       .rename(columns={2017: "pop_2017", 2025: "pop_2025"})
       .reset_index()
)
matched["category"] = pd.cut(matched["number"], bins=[0, 52, 201, 264], labels=[1, 2, 3]).astype(int)
features = matched[["number", "mono_name", "region", "category", "note", "ref_type", "oktmo"]].merge(
    pop_wide, on="oktmo", how="left"
)
features["pop_change_pct"] = (features["pop_2025"] / features["pop_2017"] - 1) * 100

print(features[["pop_2017", "pop_2025"]].isna().sum())
missing_pop = features["pop_2017"].isna() | features["pop_2025"].isna()
features.loc[missing_pop, ["mono_name", "region", "note", "ref_type", "oktmo", "pop_2017", "pop_2025"]]

pop_2017    71
pop_2025     7
dtype: int64


,mono_name,region,note,ref_type,oktmo,pop_2017,pop_2025
1,Онежский муниципальный округ Архангельской обл...,Архангельская область,automatic,Муниципальный округ,11546000,NaN,22829.0
4,Петровск-Забайкальский муниципальный округ Заб...,Забайкальский край,automatic,Муниципальный округ,76536000,NaN,28708.0
9,Междуреченский муниципальный округ Кемеровской...,Кемеровская область - Кузбасс,automatic,Муниципальный округ,32517000,NaN,95550.0
13,Топкинский муниципальный округ Кемеровской обл...,Кемеровская область - Кузбасс,automatic,Муниципальный округ,32531000,NaN,40199.0
16,Муниципальное образование Мантуровский муницип...,Костромская область,automatic,Муниципальный округ,34520000,NaN,14900.0
...,...,...,...,...,...,...,...
219,Атяшевское городское поселение Атяшевского мун...,Республика Мордовия,automatic,Городское поселение,89607151,NaN,6650.0
226,Касимовский муниципальный округ Рязанской области,Рязанская область,automatic,Муниципальный округ,61508000,NaN,53657.0
236,"Муниципальное образование ""Дорогобужский муниц...",Смоленская область,automatic,Муниципальный округ,66514000,NaN,23189.0
237,Городской округ город Невинномысск Ставропольс...,Ставропольский край,automatic,"Городской округ, городской округ с внутригород...",07724000,NaN,113112.0


In [6]:
codes = pop[["oktmo", "oktmo_stable"]].drop_duplicates()
check = features[["mono_name", "oktmo"]].merge(codes, on="oktmo", how="left")

print("towns with oktmo_stable different from oktmo:", (check["oktmo"] != check["oktmo_stable"]).sum())
check[check["oktmo"] != check["oktmo_stable"]]

towns with oktmo_stable different from oktmo: 106


,mono_name,oktmo,oktmo_stable
3,"Муниципальное образование ""Теплоозерское город...",99620170,CD
14,Муниципальное образование Демьяновское городск...,33632154,CD
17,Енисейский муниципальный округ Красноярского края,04712000,04515000
18,Парфинский муниципальный округ Новгородской об...,49630000,49530000
19,Городское поселение рабочий поселок Горный Тог...,50652153,CD
...,...,...,...
232,Малышевский муниципальный округ Свердловской о...,65762000,65535000
233,Муниципальный округ Ревда Свердловской области,65719000,65525000
234,Серовский муниципальный округ Свердловской обл...,65756000,65549000
235,Муниципальный округ Первоуральск Свердловской ...,65753000,65524000


In [7]:
def find(root):
    rows = pop["municipality"].str.contains(root, case=False, na=False)
    return pop.loc[rows, ["year", "municipality", "mun_type", "oktmo", "oktmo_stable", "indicator_value"]].sort_values(["year", "oktmo"])

for root in ["Алатыр", "Мантуров", "Краснокаменск", "Невинномысск", "Рузаев"]:
    print(f"\n=== {root} ===")
    print(find(root).to_string(index=False))


=== Алатыр ===
 year                   municipality                                                    mun_type    oktmo oktmo_stable  indicator_value
 2017 Алатырский муниципальный район                                         Муниципальный район 97603000     97503000          14875.0
 2017                  Город Алатырь Городской округ, городской округ с внутригородским делением 97704000     97503000          35298.0
 2022 Алатырский муниципальный район                                         Муниципальный район 97603000     97503000          12973.0
 2022                  Город Алатырь Городской округ, городской округ с внутригородским делением 97704000     97503000          32319.0
 2023 Алатырский муниципальный округ                                         Муниципальный округ 97503000     97503000          13552.0
 2023                  Город Алатырь Городской округ, городской округ с внутригородским делением 97704000     97503000          31550.0
 2025 Алатырский муниципальный о

## Population and population change 2017-2025

Population is joined to the monotowns by OKTMO and change is calculated in percent
(-15 = the town lost 15% of its residents). Many municipalities were reorganised between 2017 and 2025,
so gaps are filled in three steps (column `pop_source` records which one was used):

1. **Chuvashia fix:** Alatyr, Kanash and Shumerlya are matched to the town itself, because on
   1 January 2025 the municipal okrug code covered only the rural district (same approach as
   "city part of merged okrug" in notebook 02).
2. **Changed codes:** if a code has no 2017 value, 2017 population is looked up through `oktmo_stable`,
   which links old and new codes of the same territory. Merged units (e.g. town + district) are summed.
3. **District proxy:** Mordovian settlements have no 2017 data, so their district's change is used instead.

Remaining gaps (Zabaikalsky krai and Nevinnomyssk have no 2017 data; Crimea, Dagestan, Buryatia and
Tolyatti have no population data) are handled after all variables are built.
Note: 2017 estimates are based on the 2010 census and 2025 estimates on the 2021 census, so part of the
change reflects census corrections, not only real population loss.

In [8]:
#step 1: match the three Chuvash monotowns to the town instead of the okrug
for root, town in [("Алатыр", "Город Алатырь"), ("Канаш", "Город Канаш"), ("Шумерл", "Город Шумерля")]:
    town_code = pop.loc[(pop["year"] == 2025) & (pop["municipality"] == town), "oktmo"].iloc[0]
    rows = matched["mono_name"].str.contains(root)
    matched.loc[rows, ["oktmo", "note", "ref_type"]] = [town_code, "city part of merged okrug",
                                                        "Городской округ, городской округ с внутригородским делением"]

print("duplicated oktmo:", matched["oktmo"].duplicated().sum())
matched.loc[matched["region"].str.contains("Чуваш"), ["mono_name", "oktmo", "note", "ref_type"]]

duplicated oktmo: 0


,mono_name,oktmo,note,ref_type
194,Алатырский муниципальный округ Чувашской Респу...,97704000,city part of merged okrug,"Городской округ, городской округ с внутригород..."
195,Канашский муниципальный округ Чувашской Респуб...,97707000,city part of merged okrug,"Городской округ, городской округ с внутригород..."
196,Шумерлинский муниципальный округ Чувашской Рес...,97713000,city part of merged okrug,"Городской округ, городской округ с внутригород..."


In [9]:
pop_wide = (
    pop.pivot(index="oktmo", columns="year", values="indicator_value")
       .rename(columns={2017: "pop_2017", 2022: "pop_2022", 2023: "pop_2023", 2025: "pop_2025"})
)

features = matched[["number", "mono_name", "region", "category", "note", "ref_type", "oktmo"]].merge(
    pop_wide, left_on="oktmo", right_index=True, how="left"
)
features["pop_source"] = np.where(features["pop_2017"].notna() & features["pop_2025"].notna(), "direct", None)
features["pop23_source"] = np.where(features["pop_2023"].notna() & features["pop_2025"].notna(), "direct", None)

#step 2: missing 2017, 2022 and 2023 values through oktmo_stable (summing merged units)
stable_2025 = pop.loc[pop["year"] == 2025].set_index("oktmo")["oktmo_stable"]

for year, col, src in [(2017, "pop_2017", "pop_source"),
                       (2022, "pop_2022", None),
                       (2023, "pop_2023", "pop23_source")]:
    by_stable = (
        pop.loc[(pop["year"] == year) & (pop["oktmo_stable"] != "CD")]
           .groupby("oktmo_stable")["indicator_value"].sum()
    )
    gap = features[col].isna()
    features.loc[gap, col] = features.loc[gap, "oktmo"].map(stable_2025).map(by_stable)
    if src:
        features.loc[gap & features[col].notna(), src] = f"{year} via oktmo_stable"

features["pop_change_pct"] = (features["pop_2025"] / features["pop_2017"] - 1) * 100
features["pop_change_23_25"] = (features["pop_2025"] / features["pop_2023"] - 1) * 100

# census correction: jump between the last old-basis (2022) and first census-based (2023) estimate
features["census_jump_pct"] = (features["pop_2023"] / features["pop_2022"] - 1) * 100

#step 3: settlements without 2017 or 2023 data get their district's change
#district 2025 = sum of its settlements (same first 5 digits of OKTMO)
p25 = pop.loc[(pop["year"] == 2025) & ~pop["oktmo"].str.endswith("000")]
district_2025 = p25.groupby(p25["oktmo"].str[:5] + "000")["indicator_value"].sum()

for base, change, src in [("pop_2017", "pop_change_pct", "pop_source"),
                          ("pop_2023", "pop_change_23_25", "pop23_source")]:
    district_change = (district_2025 / pop_wide[base] - 1) * 100
    gap = features[change].isna() & features["pop_2025"].notna()
    features.loc[gap, change] = (features.loc[gap, "oktmo"].str[:5] + "000").map(district_change)
    features.loc[gap & features[change].notna(), src] = "district change (proxy)"

In [10]:
#step 4: a 2023 value summed via oktmo_stable is incomplete if some old units are missing in 2023
#(Kasimov: the town itself is not published for 2023, only the district)
#towns matched as "city part of merged okrug" use the town alone, so the rule does not apply to them
def units(year):
    p = pop.loc[(pop["year"] == year) & (pop["oktmo_stable"] != "CD")]
    return p.groupby("oktmo_stable")["oktmo"].nunique()

stable_code = features["oktmo"].map(stable_2025)
incomplete = ((features["pop23_source"] == "2023 via oktmo_stable")
              & (features["note"] != "city part of merged okrug")
              & (stable_code.map(units(2023)) < stable_code.map(units(2022))))

print(features.loc[incomplete, ["number", "mono_name", "pop_2022", "pop_2023", "pop_2025"]])

features.loc[incomplete, ["pop_2023", "pop_change_23_25", "census_jump_pct"]] = np.nan
features.loc[incomplete, "pop23_source"] = "missing: part of territory not published in 2023"

     number                                          mono_name  pop_2022  \
242     243  Касимовский муниципальный округ Рязанской области   55760.0   

     pop_2023  pop_2025  
242   27244.0   53657.0  


In [11]:
# how was 2023 obtained, and how do the two population changes compare?
print(features["pop23_source"].value_counts(dropna=False), "\n")
print(features[["pop_change_pct", "pop_change_23_25", "census_jump_pct"]].describe().round(1), "\n")
print("Spearman correlation:",
      round(features["pop_change_pct"].corr(features["pop_change_23_25"], method="spearman"), 2))

# towns still missing the new variable, and Slantsy for comparison
features.loc[features["pop_change_23_25"].isna() | (features["number"] == 109),
             ["number", "mono_name", "pop_2017", "pop_2022", "pop_2023", "pop_2025",
              "pop_change_pct", "pop_change_23_25", "census_jump_pct"]]

pop23_source
direct                                              197
2023 via oktmo_stable                                35
NaN                                                   7
district change (proxy)                               7
missing: part of territory not published in 2023      1
Name: count, dtype: int64 

       pop_change_pct  pop_change_23_25  census_jump_pct
count           234.0             239.0            237.0
mean             -9.1              -1.9             -2.1
std               7.9               1.3              5.9
min             -31.1              -6.3            -23.3
25%             -13.0              -2.7             -3.7
50%              -9.0              -1.9             -1.3
75%              -5.5              -1.3             -0.5
max              37.3               4.6             45.1 

Spearman correlation: 0.55


,number,mono_name,pop_2017,pop_2022,pop_2023,pop_2025,pop_change_pct,pop_change_23_25,census_jump_pct
31,32,"Городской округ ""город Дагестанские Огни""",29238.0,31472.0,31894.0,NaN,NaN,NaN,1.340874
37,38,Муниципальное образование муниципальный округ ...,24388.0,22854.0,22479.0,NaN,NaN,NaN,-1.640851
44,45,Городской округ Тольятти Самарской области,710567.0,685619.0,674630.0,NaN,NaN,NaN,-1.602785
108,109,Сланцевское городское поселение Сланцевского м...,33745.0,32833.0,35258.0,34127.0,1.132020,-3.207783,7.385862
139,140,"Городское поселение ""Каменское"" Кабанского рай...",NaN,NaN,6986.0,NaN,NaN,NaN,NaN
140,141,"Городское поселение ""Селенгинское"" Кабанского ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
141,142,"Городской округ ""город Каспийск""",113348.0,122332.0,125747.0,NaN,NaN,NaN,2.791584
145,146,Муниципальное образование городской округ Крас...,25769.0,25644.0,25381.0,NaN,NaN,NaN,-1.025581
242,243,Касимовский муниципальный округ Рязанской области,56884.0,55760.0,NaN,53657.0,-5.672948,NaN,NaN


## Net migration per 1,000 residents (average 2021-2024)

Net migration is arrivals minus departures; a negative value means more people leave than arrive.
Values are averaged over 2021-2023 (2024 is not used: the dataset contains only 10 municipalities for that year) to smooth out single unusual years, then divided by the 2025
population and multiplied by 1,000 so towns of different size can be compared.
Population for each migration year is not available, so the 2025 estimate is used as the denominator.

In [12]:
mig = data["migration"].query("year <= 2023")  # 2024 has only 10 rows: not yet published


# direct match by OKTMO
mig_avg = mig.groupby("oktmo")["indicator_value"].agg(mig_avg="mean", mig_years="count")
features = features.drop(columns=["mig_avg", "mig_years", "mig_source"], errors="ignore").merge(
    mig_avg, left_on="oktmo", right_index=True, how="left"
)
features["mig_source"] = np.where(features["mig_avg"].notna(), "direct", None)

# same territory through oktmo_stable: per year, sum old units (before a merger) or the new okrug (after)
mig_by_stable = (
    mig.loc[mig["oktmo_stable"] != "CD"]
       .groupby(["oktmo_stable", "year"])["indicator_value"].sum()
       .groupby("oktmo_stable").agg(["mean", "count"])
)
town_stable = features["oktmo"].map(stable_2025)
stable_years = town_stable.map(mig_by_stable["count"])

# use it only when it covers more years (protects towns that are part of a larger okrug)
use_stable = stable_years > features["mig_years"].fillna(0)
features.loc[use_stable, "mig_avg"] = town_stable[use_stable].map(mig_by_stable["mean"])
features.loc[use_stable, "mig_years"] = stable_years[use_stable]
features.loc[use_stable, "mig_source"] = "via oktmo_stable"

features["net_mig_per_1000"] = features["mig_avg"] / features["pop_2025"] * 1000

print(features["mig_source"].value_counts(dropna=False).to_dict())
print("years available:", features["mig_years"].value_counts(dropna=False).to_dict(), "\n")
print(features["net_mig_per_1000"].describe().round(1))

cols = ["mono_name", "mig_source", "mig_years", "mig_avg", "pop_2025", "net_mig_per_1000"]
ranked = features.dropna(subset=["net_mig_per_1000"]).sort_values("net_mig_per_1000")
display(pd.concat([ranked.head(5), ranked.tail(5)])[cols])
features.loc[features["mig_avg"].isna(), ["mono_name", "region", "note", "oktmo"]]

{'direct': 196, 'via oktmo_stable': 51}
years available: {3.0: 243, 2.0: 2, 1.0: 2} 

count    240.0
mean      -1.6
std        6.2
min      -30.0
25%       -4.8
50%       -1.4
75%        1.5
max       34.7
Name: net_mig_per_1000, dtype: float64


,mono_name,mig_source,mig_years,mig_avg,pop_2025,net_mig_per_1000
114,Печенгский муниципальный округ Мурманской области,direct,3.0,-868.333333,28918.0,-30.027434
212,Муниципальное образование Верхнекамский муници...,via oktmo_stable,3.0,-383.666667,19134.0,-20.051566
118,Окуловский муниципальный округ Новгородской об...,direct,3.0,-282.000000,17640.0,-15.986395
1,Онежский муниципальный округ Архангельской обл...,via oktmo_stable,3.0,-364.666667,22829.0,-15.973834
103,Кежемский муниципальный округ Красноярского края,direct,3.0,-221.666667,16236.0,-13.652788
61,Погарское городское поселение Погарского муниц...,direct,3.0,94.333333,9388.0,10.048289
91,Дзержинский муниципальный округ Калужской области,direct,3.0,575.000000,56006.0,10.266757
172,Городской округ Верхняя Пышма Свердловской обл...,direct,3.0,2209.000000,92113.0,23.981414
153,"Городское поселение ""Город Удачный"" муниципаль...",direct,3.0,400.333333,13957.0,28.683337
90,Боровский муниципальный округ Калужской области,direct,3.0,2937.333333,84597.0,34.721483


,mono_name,region,note,oktmo


## Average monthly wage 2025

Average monthly wage of employees of large and medium organisations (small businesses are not
included in this Rosstat indicator). Wages are mostly published for districts and okrugs, so for
urban settlements without their own value the wage of their district is used
(district OKTMO = first 5 digits of the settlement code + "000"). `wage_source` records which was used.

In [13]:
w = data["wages"].query("year == 2025").set_index("oktmo")

# regional level = median wage of all districts and okrugs in the region (codes ending in "000")
region_median = w[w.index.str.endswith("000")].groupby("region_name")["indicator_value"].median()
w["wage_rel"] = w["indicator_value"] / w["region_name"].map(region_median)

# stable code -> wage row, only if exactly one unit has that stable code
by_stable = w[w["oktmo_stable"] != "CD"].reset_index().groupby("oktmo_stable")["oktmo"]
stable_to_row = by_stable.first()[by_stable.size() == 1]

# wage row for each town: own code, else its district, else the same territory under a new code
own = features["oktmo"].where(features["oktmo"].isin(w.index))
district = (features["oktmo"].str[:5] + "000").where(lambda s: s.isin(w.index))
stable = features["oktmo"].map(stable_2025).map(stable_to_row)

features["wage_key"] = own.fillna(district).fillna(stable)
features["wage_source"] = np.where(own.notna(), "direct",
                          np.where(district.notna(), "district",
                          np.where(stable.notna(), "via oktmo_stable", None)))
features["wage_2025"] = features["wage_key"].map(w["indicator_value"])
features["wage_rel"] = features["wage_key"].map(w["wage_rel"])

print(features["wage_source"].value_counts(dropna=False).to_dict(), "\n")
print(features[["wage_2025", "wage_rel"]].describe().round(2))

cols = ["mono_name", "region", "wage_source", "wage_2025", "wage_rel"]
ranked = features.dropna(subset=["wage_rel"]).sort_values("wage_rel")
display(pd.concat([ranked.head(5), ranked.tail(5)])[cols])
features.loc[features["wage_2025"].isna(), ["mono_name", "region", "note", "oktmo"]]

{'direct': 168, 'district': 57, 'via oktmo_stable': 19, nan: 3} 

       wage_2025  wage_rel
count     244.00    244.00
mean    80611.92      1.14
std     26388.44      0.25
min     36825.70      0.70
25%     64176.48      0.99
50%     75486.75      1.08
75%     89495.78      1.25
max    217045.60      2.85


,mono_name,region,wage_source,wage_2025,wage_rel
77,Агинский муниципальный округ Забайкальского края,Забайкальский край,direct,62556.8,0.697426
156,"Городское поселение ""Поселок Мохсоголлох"" муни...",Республика Саха (Якутия),district,96124.8,0.729822
0,Городской округ город Райчихинск Амурской области,Амурская область,direct,73254.7,0.769840
50,Нязепетровский муниципальный округ Челябинской...,Челябинская область,direct,52419.8,0.778419
66,Меленковский муниципальный округ Владимирской ...,Владимирская область,direct,57735.0,0.790425
90,Боровский муниципальный округ Калужской области,Калужская область,direct,113210.0,1.664471
189,Пластовский муниципальный округ Челябинской об...,Челябинская область,direct,114720.5,1.703566
62,Сельцовский городской округ Брянской области,Брянская область,direct,117880.7,2.021104
217,Городской округ город Норильск Красноярского края,Красноярский край,direct,200213.3,2.659205
157,Город Елабуга Елабужского муниципального район...,Республика Татарстан,district,217045.6,2.845929


,mono_name,region,note,oktmo
139,"Городское поселение ""Каменское"" Кабанского рай...",Республика Бурятия,matched via migration table,81624158
140,"Городское поселение ""Селенгинское"" Кабанского ...",Республика Бурятия,matched via migration table,81624163
253,Городской округ город Невинномысск Ставропольс...,Ставропольский край,automatic,07724000


## Jobs per 1,000 residents 2025 and job change 2017-2025

Jobs = average number of employees of large and medium organisations (small businesses are not included,
so absolute levels are underestimated, but towns remain comparable). Employees come from the same unit as
the wage (`wage_key`: the town, its district, or the same territory under a new code), and are divided by the
2025 population of that same unit. Job change compares 2025 with 2017; if the unit did not exist in 2017,
the old units of the same territory are summed through `oktmo_stable`, as for population.

In [14]:
emp = data["employees"]
e25 = emp.query("year == 2025").set_index("oktmo")
e17 = emp.query("year == 2017")
e17_own = e17.set_index("oktmo")["indicator_value"]
e17_stable = e17[e17["oktmo_stable"] != "CD"].groupby("oktmo_stable")["indicator_value"].sum()

key = features["wage_key"]
features["emp_2025"] = key.map(e25["indicator_value"])

# 2017: same code, else old units of the same territory summed
features["emp_2017"] = key.map(e17_own)
gap = features["emp_2017"].isna()
features.loc[gap, "emp_2017"] = key[gap].map(e25["oktmo_stable"]).map(e17_stable)
features["job_change_pct"] = (features["emp_2025"] / features["emp_2017"] - 1) * 100

# denominator: 2025 population of the same unit the jobs come from
pop25 = pop.query("year == 2025")
pop25_own = pop25.set_index("oktmo")["indicator_value"]
pop25_stable = pop25[pop25["oktmo_stable"] != "CD"].groupby("oktmo_stable")["indicator_value"].sum()

denom = features["pop_2025"].copy()
is_district = features["wage_source"] == "district"
denom[is_district] = key[is_district].map(pop25_own).fillna(key[is_district].map(district_2025))
is_stable = features["wage_source"] == "via oktmo_stable"
denom[is_stable] = key[is_stable].map(e25["oktmo_stable"]).map(pop25_stable)

features["pop_jobs_unit"] = denom
features["jobs_per_1000"] = features["emp_2025"] / denom * 1000

print(features[["emp_2025", "emp_2017", "jobs_per_1000", "job_change_pct"]].isna().sum().to_dict(), "\n")
print(features[["jobs_per_1000", "job_change_pct"]].describe().round(1))

cols = ["mono_name", "wage_source", "emp_2017", "emp_2025", "pop_jobs_unit", "jobs_per_1000", "job_change_pct"]
for var in ["jobs_per_1000", "job_change_pct"]:
    ranked = features.dropna(subset=[var]).sort_values(var)
    display(pd.concat([ranked.head(5), ranked.tail(5)])[cols])

{'emp_2025': 3, 'emp_2017': 9, 'jobs_per_1000': 8, 'job_change_pct': 9} 

       jobs_per_1000  job_change_pct
count          239.0           238.0
mean           236.1            -5.4
std             92.0            18.6
min             61.4           -42.4
25%            171.3           -14.7
50%            218.9            -8.6
75%            268.3            -0.6
max            618.3           101.8


,mono_name,wage_source,emp_2017,emp_2025,pop_jobs_unit,jobs_per_1000,job_change_pct
79,Колобовское городское поселение Шуйского муниц...,district,1300.0,1162.0,18930.0,61.384046,-10.615385
39,"Муниципальное образование городской округ ""Гор...",direct,7521.0,5280.0,58316.0,90.541189,-29.796570
40,"Муниципальное образование городской округ ""Гор...",direct,5242.0,4362.0,45334.0,96.219173,-16.787486
211,Медногорское городское поселение Урупского мун...,district,2268.0,2350.0,23937.0,98.174374,3.615520
83,Южское городское поселение Южского муниципальн...,district,2342.0,1868.0,18832.0,99.192863,-20.239112
152,"Городское поселение ""Город Нерюнгри"" муниципал...",district,23344.5,37799.0,68623.0,550.821153,61.918225
153,"Городское поселение ""Город Удачный"" муниципаль...",district,36923.3,40489.0,72132.0,561.318139,9.657046
154,"Городское поселение ""Поселок Айхал"" муниципаль...",district,36923.3,40489.0,72132.0,561.318139,9.657046
155,"Городское поселение ""Город Мирный"" муниципальн...",district,36923.3,40489.0,72132.0,561.318139,9.657046
112,Муниципальное образование муниципальный округ ...,direct,13424.0,15925.0,25756.0,618.302531,18.630810


,mono_name,wage_source,emp_2017,emp_2025,pop_jobs_unit,jobs_per_1000,job_change_pct
35,"Муниципальный округ ""Инта"" Республики Коми",direct,8937.0,5146.0,20198.0,254.777701,-42.419156
47,"""Инзенское городское поселение"" Инзенского рай...",district,6393.0,3704.0,25422.0,145.700574,-42.061630
212,Муниципальное образование Верхнекамский муници...,direct,6519.0,4139.0,19134.0,216.316505,-36.508667
30,Печорский муниципальный округ Псковской области,direct,3918.0,2717.0,17858.0,152.144697,-30.653395
39,"Муниципальное образование городской округ ""Гор...",direct,7521.0,5280.0,58316.0,90.541189,-29.796570
240,"Муниципальное образование ""город Зеленодольск""...",district,35293.0,58112.0,170312.0,341.209075,64.655881
157,Город Елабуга Елабужского муниципального район...,district,25371.0,43404.0,86096.0,504.134919,71.077214
62,Сельцовский городской округ Брянской области,direct,2675.0,4836.0,15207.0,318.011442,80.785047
254,Городской округ город Котовск Тамбовской области,direct,6227.0,11804.0,25852.0,456.599103,89.561587
180,Городской округ город Алексин Тульской области,direct,14843.0,29958.0,68427.0,437.809637,101.832514


## Industry dependence 2025

Share of jobs in mining (OKVED section B) and manufacturing (section C) among all jobs of large and
medium organisations, for the same unit as wages and jobs. Rosstat does not publish an industry when
very few companies work in it (confidentiality), and such rows are simply missing. In monotowns this
often hides the main employer: Vorkuta, a coal-mining town, shows no mining jobs. A missing sector is
therefore counted as 0, and the result `industry_share_min` is a lower bound. `industry_complete`
marks towns where both sectors are published and the share is reliable. Both sectors are published for only 40 of 247 towns, so industry dependence is not used for clustering; it is kept only to describe the clusters.

In [15]:
ind = data["industry"].query("year == 2025").copy()
ind["sector"] = np.where(ind["okved2"].str.contains("Добыча"), "mining", "manufacturing")

values = ind.pivot(index="oktmo", columns="sector", values="indicator_value")
has_row = ind.assign(present=True).pivot(index="oktmo", columns="sector", values="present")
values = values.fillna(0)  # absent sector counted as 0, so the share is a lower bound
complete = has_row.notna().all(axis=1)  # both sectors published

key = features["wage_key"]
features["emp_mining"] = key.map(values["mining"])
features["emp_manuf"] = key.map(values["manufacturing"])
features["industry_share_min"] = (features["emp_mining"] + features["emp_manuf"]) / features["emp_2025"] * 100
features["industry_complete"] = key.map(complete)

print("unit not in industry file:", (key.notna() & ~key.isin(values.index)).sum())
print("both sectors published:", features["industry_complete"].value_counts(dropna=False).to_dict(), "\n")
print(features.groupby("industry_complete")["industry_share_min"].describe().round(1))

unit not in industry file: 59
both sectors published: {False: 145, nan: 62, True: 40} 

                   count  mean   std  min   25%   50%   75%   max
industry_complete                                                
False              145.0  31.2  13.8  1.5  21.3  32.7  40.2  65.1
True                40.0  36.2  12.0  7.5  29.6  35.4  42.4  63.1


## English names

Town and region names come from a hand-made lookup `data/names_en.csv` (joined by item number),
because automatic transliteration fails on official names ("Онежский муниципальный округ" is the
town of Onega). Towns keep their own name; municipal okrugs are named after their administrative
centre. Municipality types are translated: urban okrug (a town governed as its own municipality),
municipal okrug (a town and surrounding villages without lower-level settlements),
municipal district (a district containing settlements), urban settlement (a town inside a district).

In [16]:
names_en = pd.read_csv("../data/names_en.csv")
features = features.drop(columns=["name_en", "region_en"], errors="ignore").merge(names_en, on="number", how="left")

type_en = {
    "Городской округ, городской округ с внутригородским делением": "urban okrug",
    "Муниципальный округ": "municipal okrug",
    "Муниципальный район": "municipal district",
    "Городское поселение": "urban settlement",
}
features["type_en"] = features["ref_type"].map(type_en)

print(features[["name_en", "region_en", "type_en"]].isna().sum().to_dict())
features.loc[features["number"] == 109, ["number", "name_en", "region_en", "type_en", "category"]]

{'name_en': 0, 'region_en': 0, 'type_en': 0}


,number,name_en,region_en,type_en,category
98,109,Slantsy,Leningrad Oblast,urban settlement,2


## Missing values and output

Clustering uses five variables: population change, net migration per 1,000, relative wage, jobs per 1,000
and job change. Towns missing any of them are kept in the output but marked `in_clustering = False`
instead of filling gaps with estimated values, which would place them in arbitrary clusters.
They are listed in the README limitations. The result is saved to `data/processed/features.csv`.

In [17]:
cluster_vars = ["pop_change_pct", "net_mig_per_1000", "wage_rel", "jobs_per_1000", "job_change_pct"]

print(features[cluster_vars].isna().sum().to_dict())
features["in_clustering"] = features[cluster_vars].notna().all(axis=1)
print(features["in_clustering"].value_counts().to_dict())

features.loc[~features["in_clustering"], ["number", "name_en", "region_en", "note"] + cluster_vars]

{'pop_change_pct': 13, 'net_mig_per_1000': 7, 'wage_rel': 3, 'jobs_per_1000': 8, 'job_change_pct': 9}
{True: 228, False: 19}


,number,name_en,region_en,note,pop_change_pct,net_mig_per_1000,wage_rel,jobs_per_1000,job_change_pct
4,9,Petrovsk-Zabaykalsky,Zabaykalsky Krai,automatic,NaN,-7.535646,0.859121,213.703497,-11.675785
23,32,Dagestanskiye Ogni,Republic of Dagestan,matched via migration table,NaN,NaN,0.892894,NaN,-1.480485
29,38,Armyansk,Crimea,matched via migration table,NaN,NaN,1.071084,NaN,-25.562781
36,45,Tolyatti,Samara Oblast,matched via migration table,NaN,NaN,1.398987,NaN,-6.756014
66,76,Verkh-Usugli,Zabaykalsky Krai,automatic,NaN,-12.963759,1.037182,192.629996,-13.715111
67,77,Krasnokamensk,Zabaykalsky Krai,automatic,NaN,4.438030,0.990284,255.583619,-12.316643
68,78,Aginskoye,Zabaykalsky Krai,automatic,NaN,-10.614753,0.697426,119.512667,-23.640857
69,79,Borzya,Zabaykalsky Krai,automatic,NaN,-11.055068,1.000000,189.985951,-11.680942
128,140,Kamensk,Republic of Buryatia,matched via migration table,NaN,NaN,NaN,NaN,NaN
129,141,Selenginsk,Republic of Buryatia,matched via migration table,NaN,NaN,NaN,NaN,NaN


In [18]:
out_cols = [
    "number", "name_en", "mono_name", "region_en", "region", "category", "type_en", "note", "oktmo",
    "pop_2017", "pop_2022", "pop_2023", "pop_2025",
    "pop_change_pct", "pop_source", "pop_change_23_25", "pop23_source", "census_jump_pct",
    "mig_avg", "mig_years", "mig_source", "net_mig_per_1000",
    "wage_2025", "wage_rel", "wage_source", "wage_key",
    "emp_2017", "emp_2025", "jobs_per_1000", "job_change_pct", "pop_jobs_unit",
    "emp_mining", "emp_manuf", "industry_share_min", "industry_complete",
    "in_clustering",
]
features[out_cols].to_csv(PROCESSED / "features.csv", index=False, encoding="utf-8-sig")
print(features[out_cols].shape)

(247, 36)


In [19]:
# temporary: look at the extremes
cols = ["number", "mono_name", "pop23_source", "pop_2022", "pop_2023", "pop_2025",
        "pop_change_23_25", "census_jump_pct"]
display(features.sort_values("pop_change_23_25", ascending=False)[cols].head(5))
display(features.sort_values("census_jump_pct")[cols].head(3))
display(features.sort_values("census_jump_pct")[cols].tail(3))

,number,mono_name,pop23_source,pop_2022,pop_2023,pop_2025,pop_change_23_25,census_jump_pct
142,154,"Городское поселение ""Город Удачный"" муниципаль...",direct,12918.0,13349.0,13957.0,4.554648,3.336430
160,173,Городской округ Верхняя Пышма Свердловской обл...,direct,90210.0,88206.0,92113.0,4.429404,-2.221483
81,91,Боровский муниципальный округ Калужской области,direct,77154.0,82143.0,84597.0,2.987473,6.466288
17,23,Енисейский муниципальный округ Красноярского края,direct,17469.0,17655.0,17868.0,1.206457,1.064743
143,155,"Городское поселение ""Поселок Айхал"" муниципаль...",direct,13655.0,13359.0,13487.0,0.958156,-2.167704


,number,mono_name,pop23_source,pop_2022,pop_2023,pop_2025,pop_change_23_25,census_jump_pct
15,21,Муниципальное образование городской округ горо...,direct,16433.0,12600.0,12358.0,-1.920635,-23.325017
101,112,Ловозерский муниципальный округ Мурманской обл...,direct,10679.0,8695.0,8554.0,-1.621622,-18.578519
27,36,"Муниципальный округ ""Инта"" Республики Коми",2023 via oktmo_stable,25786.0,21092.0,20198.0,-4.238574,-18.203676


,number,mono_name,pop23_source,pop_2022,pop_2023,pop_2025,pop_change_23_25,census_jump_pct
152,164,Лесновское городское поселение Шиловского муни...,district change (proxy),8096.0,NaN,8052.0,-2.582505,NaN
219,236,Атяшевское городское поселение Атяшевского мун...,district change (proxy),NaN,NaN,6650.0,-3.349173,NaN
226,243,Касимовский муниципальный округ Рязанской области,missing: part of territory not published in 2023,55760.0,NaN,53657.0,NaN,NaN


In [20]:
# temporary: which Kasimov units exist in each year?
k = pop[pop["year"].isin([2022, 2023, 2025]) & pop["municipality"].str.contains("Касимов", na=False)]
k[["year", "municipality", "mun_type", "oktmo", "oktmo_stable", "indicator_value"]].sort_values(["year", "oktmo"])

,year,municipality,mun_type,oktmo,oktmo_stable,indicator_value
26817,2022,Касимовский муниципальный район,Муниципальный район,61608000,61508000,27563.0
27083,2022,Город Касимов,"Городской округ, городской округ с внутригород...",61705000,61508000,28197.0
42051,2023,Касимовский муниципальный район,Муниципальный район,61608000,61508000,27244.0
56072,2025,Касимовский муниципальный округ,Муниципальный округ,61508000,61508000,53657.0
